# Surface Code QEC with Stim & Sinter in Maestro 0.3.1

This notebook demonstrates Maestro 0.3.1's native **Stim and Sinter integration** () for quantum error correction (QEC) benchmarking with PyMatching.

* **Act 1 — The Drop-In QEC Engine**: Generate rotated surface code circuits with Stim, benchmark decoding performance with Sinter, and use Maestro's Matrix Product State () as a custom sampler backend.
* **Act 2 — Beyond-Clifford Noise**: Real hardware experiences coherent over-rotations and idle dephasing. Stim cannot model non-Clifford rotations, but Maestro's tensor-network engine does, revealing how physical noise shifts the error threshold.

In [ ]:
import os, sys
if os.path.isdir('surface_code_noise') and 'surface_code_noise' not in sys.path:
    sys.path.insert(0, os.path.abspath('surface_code_noise'))
import time
import numpy as np
import matplotlib.pyplot as plt
import stim
import sinter
import pymatching
import maestro
from maestro.sinter import MaestroSinterSampler
from qec_plotting import plot_threshold_curves, plot_noise_comparison

print(f"Stim version:       {stim.__version__}")
print(f"Sinter version:     {sinter.__version__}")
print(f"PyMatching version: {pymatching.__version__}")
print(f"Maestro GPU Avail:  {maestro.is_gpu_available()}")


---
## Act 1: Drop-In Sinter QEC Benchmarking with Maestro MPS

We define rotated surface code memory tasks across code distances =3$ and =5$ over a sweep of physical error rates using Stim's circuit generator, and collect decoding statistics via Sinter.

In [ ]:
distances = [3, 5]
physical_error_rates = [0.005, 0.01, 0.015, 0.02]
shots = 500
chi = 32

tasks = []
for d in distances:
    for p in physical_error_rates:
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            distance=d,
            rounds=d,
            after_clifford_depolarization=p,
        )
        tasks.append(sinter.Task(circuit=circuit, json_metadata={"d": d, "p": p, "sampler": "maestro"}))

custom_decoders = {"maestro": MaestroSinterSampler(chi=chi)}

print("Collecting Sinter samples with Maestro MPS sampler...")
t0 = time.perf_counter()
stats = sinter.collect(
    num_workers=1,
    max_shots=shots,
    tasks=tasks,
    decoders=["pymatching"],
    custom_decoders=custom_decoders,
    print_progress=False,
)
print(f"✓ Completed in {time.perf_counter() - t0:.2f}s")
print()

print(f"{'Distance':<10} {'Physical Err':<14} {'Shots':<10} {'Errors':<10} {'Logical P_L':<14}")
print("-" * 58)
for s in stats:
    d = s.json_metadata["d"]
    p = s.json_metadata["p"]
    p_l = s.errors / max(1, s.shots)
    print(f"d = {d:<6} {p:<14.4f} {s.shots:<10} {s.errors:<10} {p_l:<14.6f}")


### Act 1 Threshold Visualization
Plot the logical error rate vs physical error rate curves for distances =3$ and =5$.

In [ ]:
plot_threshold_curves(stats, "qec_threshold_curve.png")

img = plt.imread("qec_threshold_curve.png")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

---
## Act 2: Beyond-Clifford Noise (Coherent & Idle Simulation)

Stim's stabilizer tableau simulator cannot simulate continuous rotations. With Maestro 0.3.1, we attach a  with **coherent rotation noise** and **idle decoherence** () to . Sinter and PyMatching decode the resulting syndromes, revealing the physical threshold shift.

In [ ]:
d = 3
num_qubits = 2 * (d ** 2) - 1
noise_sweep = [0.005, 0.01, 0.015, 0.02]

pauli_tasks = [
    sinter.Task(
        circuit=stim.Circuit.generated("surface_code:rotated_memory_z", distance=d, rounds=d, after_clifford_depolarization=p),
        json_metadata={"d": d, "p": p, "sampler": "pauli"}
    )
    for p in noise_sweep
]

clean_circuit = stim.Circuit.generated("surface_code:rotated_memory_z", distance=d, rounds=d)

print("Sampling Pauli depolarizing noise baseline...")
pauli_stats = sinter.collect(
    num_workers=1,
    max_shots=shots,
    tasks=pauli_tasks,
    decoders=["pymatching"],
    custom_decoders={"maestro": MaestroSinterSampler(chi=chi)},
    print_progress=False,
)

print("Sampling Coherent + Idle noise with Maestro NoiseModel...")
coherent_stats = []
for p in noise_sweep:
    nm = maestro.NoiseModel()
    nm.set_all_coherent_depolarizing(num_qubits, p)
    for q in range(num_qubits):
        nm.set_idle_noise(q, t1=100e-6, t2=50e-6)
    coh_sampler = MaestroSinterSampler(chi=chi, noise_model=nm)
    task = sinter.Task(circuit=clean_circuit, json_metadata={"d": d, "p": p, "sampler": "coherent"})
    c_stat = sinter.collect(
        num_workers=1,
        max_shots=shots,
        tasks=[task],
        decoders=["pymatching"],
        custom_decoders={"maestro": coh_sampler},
        print_progress=False,
    )
    coherent_stats.extend(c_stat)

print()
print(f"{'Phys Err (p)':<14} {'Pauli P_L':<14} {'Coherent P_L':<16}")
print("-" * 44)
for p_stat, c_stat in zip(pauli_stats, coherent_stats):
    p_val = p_stat.json_metadata["p"]
    p_l_p = p_stat.errors / max(1, p_stat.shots)
    p_l_c = c_stat.errors / max(1, c_stat.shots)
    print(f"{p_val:<14.4f} {p_l_p:<14.6f} {p_l_c:<16.6f}")


### Act 2 Noise Comparison Visualization
Visualizing the logical error rate comparison between Pauli depolarizing noise and coherent/idle noise.

In [ ]:
plot_noise_comparison(pauli_stats, coherent_stats, "qec_noise_comparison.png", distance=d)

img = plt.imread("qec_noise_comparison.png")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.show()